# CMAPSS - Exploratory Data Analysis
**Arkon Manufacturing AI | Module: Time Series | Department: Engine Testing**

NASA CMAPSS (Commercial Modular Aero-Propulsion System Simulation) dataset.
Contains run-to-failure sensor data from turbofan engines across 4 sub-datasets (FD001–FD004).

**Goal:** Understand the data structure, sensor behaviour, and engine degradation patterns
before building a Remaining Useful Life (RUL) prediction model.

---
| Sub-dataset | Operating Conditions | Fault Modes |
|-------------|---------------------|-------------|
| FD001       | 1                   | 1           |
| FD002       | 6                   | 1           |
| FD003       | 1                   | 2           |
| FD004       | 6                   | 2           |

We start with **FD001** - the simplest case.

In [ ]:
import sys
from pathlib import Path

# Add notebooks/utils to path (works on Mac, Windows, Linux)
_nb_root = Path('..').resolve()
if str(_nb_root) not in sys.path:
    sys.path.insert(0, str(_nb_root))

from utils.arkon_utils import (
    get_device, get_mlflow_uri, save_figure,
    Timer, CheckpointManager, recommended_num_workers
)

print('arkon_utils loaded ✓')

## 1. Imports & Configuration

In [1]:
# Import all libraries needed for data loading and visualisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Assets subfolder for this module
ASSETS = 'timeseries'


Data directory exists: True


## 2. Column Names

In [2]:
# Define column names manually - CMAPSS files have no header row
COLUMNS = [
    'unit',           # engine unit number (1 engine = 1 run-to-failure)
    'cycle',          # operational cycle number (time step)
    'os_1', 'os_2', 'os_3',   # 3 operational settings
    's1',  's2',  's3',  's4',  's5',  's6',  's7',
    's8',  's9',  's10', 's11', 's12', 's13', 's14',
    's15', 's16', 's17', 's18', 's19', 's20', 's21'  # 21 sensor measurements
]
print(f'Total columns: {len(COLUMNS)}')

Total columns: 26


## 3. Load Data

In [3]:
DATA_DIR   = Path('../../data/01_cmapss/raw/CMaps')
assert DATA_DIR.exists(), f'Data not found: {DATA_DIR}'
print(f'Data dir: {DATA_DIR}')


<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:12: SyntaxWarning: invalid escape sequence '\s'
/var/folders/j0/40n21yjd3cn0n5l4r27tkwy40000gn/T/ipykernel_76951/2163292357.py:4: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',
/var/folders/j0/40n21yjd3cn0n5l4r27tkwy40000gn/T/ipykernel_76951/2163292357.py:12: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',


Train shape: (20631, 26)
Test shape:  (13096, 26)
RUL shape:   (100, 1)


## 4. First Look

In [ ]:
# Show the first 5 rows to understand the data structure
df_train.head()

In [ ]:
# Show data types and memory usage - check for unexpected types
df_train.info()

In [ ]:
# Check basic statistics - min, max, mean, std for each column
df_train.describe().round(2)

## 5. Missing Values

In [ ]:
# Check for missing values - CMAPSS should have none, but always verify
missing = df_train.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✓')

## 6. How Many Engines?

In [ ]:
# Count number of unique engines and their operational lifetimes in cycles
n_engines_train = df_train['unit'].nunique()
n_engines_test  = df_test['unit'].nunique()

print(f'Engines in training set: {n_engines_train}')
print(f'Engines in test set:     {n_engines_test}')

# Show max cycle (lifespan) per engine in training set
engine_life = df_train.groupby('unit')['cycle'].max().sort_values()
print(f'\nEngine lifetime (cycles):')
print(f'  Min: {engine_life.min()}')
print(f'  Max: {engine_life.max()}')
print(f'  Mean: {engine_life.mean():.1f}')

## 7. Engine Lifetime Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
engine_life.plot(kind='hist', bins=20, ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Lifetime (cycles)')
ax.set_ylabel('Count')
ax.set_title('Engine Lifetime Distribution - FD001 Training Set')
save_figure(fig, 'cmapss_eda_engine_lifetime', subfolder=ASSETS)
plt.show()


## 8. Sensor Degradation Over Time

In [ ]:
SENSORS = ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
engine_id = 1

engine_data = df_train[df_train['unit'] == engine_id]
fig, axes = plt.subplots(len(SENSORS)//2, 2, figsize=(14, 20))
axes = axes.flatten()
for i, sensor in enumerate(SENSORS):
    axes[i].plot(engine_data['cycle'], engine_data[sensor], linewidth=0.8)
    axes[i].set_title(sensor)
    axes[i].set_xlabel('Cycle')
plt.suptitle(f'Engine {engine_id} - Sensor Readings Over Time', y=1.01)
plt.tight_layout()
save_figure(fig, 'cmapss_eda_sensor_degradation', subfolder=ASSETS)
plt.show()


## 9. Constant Sensors (Zero Variance)

In [ ]:
# Identify sensors with zero or near-zero variance - they carry no useful information
sensor_cols = [c for c in COLUMNS if c.startswith('s')]
variances = df_train[sensor_cols].var()

flat_sensors = variances[variances < 0.01].index.tolist()
useful_sensors = variances[variances >= 0.01].index.tolist()

print(f'Flat (useless) sensors: {flat_sensors}')
print(f'\nUseful sensors ({len(useful_sensors)}): {useful_sensors}')

## 10. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = df_train[useful_sensors].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax)
ax.set_title('Sensor Correlation Matrix - FD001')
save_figure(fig, 'cmapss_eda_correlation_heatmap', subfolder=ASSETS)
plt.show()


## 11. Compare All 4 Sub-datasets

In [ ]:
# Load all 4 FD datasets to compare their size and complexity
datasets = {}
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    datasets[fd] = pd.read_csv(
        DATA_DIR / f'train_{fd}.txt',
        sep=r'\s+', header=None, names=COLUMNS
    )

# Print summary statistics for each sub-dataset
print(f'{"Dataset":<8} {"Engines":<10} {"Total rows":<12} {"Max cycles":<12}')
print('-' * 45)
for name, df in datasets.items():
    n_eng = df['unit'].nunique()
    n_rows = len(df)
    max_cyc = df.groupby('unit')['cycle'].max().max()
    print(f'{name:<8} {n_eng:<10} {n_rows:<12} {max_cyc:<12}')

## 12. Multi-engine Degradation Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sample_engines = df_train['unit'].unique()[:10]
for eng in sample_engines:
    eng_data = df_train[df_train['unit'] == eng]
    ax.plot(range(len(eng_data)), eng_data['s11'], alpha=0.6, linewidth=0.8)
ax.set_xlabel('Relative cycle')
ax.set_ylabel('s11')
ax.set_title('Sensor s11 Degradation - 10 Engines')
save_figure(fig, 'cmapss_eda_multiengine_degradation', subfolder=ASSETS)
plt.show()


## 13. Key EDA Findings

| Finding | Detail |
|---------|--------|
| No missing values | Dataset is clean |
| 100 engines in FD001 train | Each runs to failure |
| Several flat sensors | s1, s5, s6, s10, s16, s18, s19 - drop in preprocessing |
| Clear degradation trends | Sensors s11, s12, s13, s14, s17, s21 show strong signal |
| Variable engine lifetime | 128–362 cycles - model must handle variable-length sequences |

**Next step:** `02_cmapss_preprocessing.ipynb` - add RUL labels, drop flat sensors, normalise.